**Imports**

In [35]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import shap

from datetime import datetime

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


**Mount Google Drive**

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Define Project Paths**

In [37]:
BASE_DIR = "/content/drive/MyDrive/c01-price-forecasting"

DATA_PATH = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "cleaned_data.csv"
)

MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "xgboost_model.pkl"
)

print("Dataset :", DATA_PATH)
print("Model   :", MODEL_PATH)

Dataset : /content/drive/MyDrive/c01-price-forecasting/data/processed/cleaned_data.csv
Model   : /content/drive/MyDrive/c01-price-forecasting/models/xgboost_model.pkl


**Load Model**

In [38]:
MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "xgboost_model.pkl"
)

model = joblib.load(MODEL_PATH)

print("Model Loaded Successfully")

Model Loaded Successfully


**Load Dataset**

In [39]:
DATA_PATH = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "cleaned_data.csv"
)

df = pd.read_csv(DATA_PATH)

print(df.shape)

display(df.head())

(2308, 30)


,date,district,paddy_type,min_price,max_price,avg_price,production_total,price_range,week_of_year,week_sin,...,year,month,week,lag_1,lag_2,lag_4,lag_12,rolling_mean_4,rolling_std_4,season_Yala
0,2015-03-26,Ampara,Long_Grain_White,34.0,40.0,37.14,307661,6.0,13,1.000000,...,2015,3,13,37.48,38.50,36.34,27.50,37.6575,0.586195,False
1,2015-04-02,Ampara,Long_Grain_White,34.0,40.0,36.40,309335,6.0,14,0.992709,...,2015,4,14,37.14,37.48,37.51,26.10,37.3800,0.872238,True
2,2015-04-09,Ampara,Long_Grain_White,34.0,39.0,35.68,309335,5.0,15,0.970942,...,2015,4,15,36.40,37.14,38.50,26.58,36.6750,0.802060,True
3,2015-04-16,Ampara,Long_Grain_White,31.0,37.0,34.97,309335,6.0,16,0.935016,...,2015,4,16,35.68,36.40,37.48,28.95,36.0475,0.933430,True
4,2015-04-23,Ampara,Long_Grain_White,30.0,38.0,34.03,309335,8.0,17,0.885456,...,2015,4,17,34.97,35.68,37.14,30.64,35.2700,1.012028,True


**Prepare Feature Matrix**

In [41]:
def prepare_feature_matrix(dataframe, model):
    """
    Prepare feature matrix exactly as used during
    XGBoost training.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Original cleaned dataset.

    model : Trained XGBoost model

    Returns
    -------
    X : Feature matrix

    y : Target variable
    """

    df_copy = dataframe.copy()

    # Target
    y = df_copy["avg_price"]

    # Remove columns not used for training
    drop_columns = [
        "date",
        "avg_price",
        "paddy_type",
        "price_t-1",
        "price_t-2",
        "price_t-3",
        "price_t-4",
        "price_t-8",
        "price_t-12"
    ]

    df_copy = df_copy.drop(columns=drop_columns)

    # One-Hot Encode district
    df_copy = pd.get_dummies(
        df_copy,
        columns=["district"],
        drop_first=True
    )

    # Get expected feature order
    feature_names = model.feature_names_in_

    # Add any missing dummy columns
    for col in feature_names:
        if col not in df_copy.columns:
            df_copy[col] = 0

    # Keep exact order used during training
    X = df_copy[feature_names]

    return X, y

In [42]:
X, y = prepare_feature_matrix(df, model)

print(X.shape)

(2308, 23)


**Create SHAP Explainer**

In [43]:
explainer = shap.TreeExplainer(model)

shap_values = explainer.shap_values(X)

print("SHAP Ready")

SHAP Ready


**Create Output Folder**

In [44]:
OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "reports",
    "phase08"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print(OUTPUT_DIR)

/content/drive/MyDrive/c01-price-forecasting/reports/phase08


**Explanation Engine Class**

In [45]:
class ExplanationEngine:

    def __init__(self, model, explainer):

        self.model = model

        self.explainer = explainer

    def predict(self, sample):

        prediction = self.model.predict(sample)[0]

        shap_value = self.explainer.shap_values(sample)[0]

        return prediction, shap_value

**Create Engine**

In [46]:
engine = ExplanationEngine(
    model,
    explainer
)

print("Explanation Engine Ready")

Explanation Engine Ready


**Test the Engine**

In [50]:
sample = X.iloc[[100]]

prediction, local_shap = engine.predict(sample)

print("Prediction")

print(prediction)

print()

print("Number of SHAP Values")

print(len(local_shap))

Prediction
37.048306

Number of SHAP Values
23


**Create Explanation Object**

In [51]:
explanation = {

    "prediction": float(prediction),

    "timestamp": datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    ),

    "feature_count": len(local_shap)

}

display(explanation)

{'prediction': 37.04830551147461,
 'timestamp': '2026-07-23 11:29:02',
 'feature_count': 23}

**Save Json**

In [52]:
OUTPUT_DIR = os.path.join(BASE_DIR, "reports", "causal_analysis")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output Directory:")
print(OUTPUT_DIR)

Output Directory:
/content/drive/MyDrive/c01-price-forecasting/reports/causal_analysis


In [53]:
with open(

    os.path.join(
        OUTPUT_DIR,
        "explanation.json"
    ),

    "w"

) as f:

    json.dump(
        explanation,
        f,
        indent=4
    )

print("Explanation JSON Saved")

Explanation JSON Saved


**Trend Classification**

In [54]:
def classify_trend(previous_price, predicted_price):

    change = predicted_price - previous_price

    percentage = (change / previous_price) * 100

    if percentage >= 5:
        return "Strong Increase"

    elif percentage >= 2:
        return "Increase"

    elif percentage > -2:
        return "Stable"

    elif percentage > -5:
        return "Decrease"

    else:
        return "Strong Decrease"

**Confidence Estimation**

In [56]:
def estimate_confidence(shap_values):

    absolute = np.abs(shap_values)

    dominance = absolute.max() / absolute.sum()

    if dominance >= 0.60:
        return "High"

    elif dominance >= 0.35:
        return "Medium"

    else:
        return "Low"

**Extract Top SHAP Features**

In [57]:
def get_top_features(sample, shap_values, top_n=3):

    explanation_df = pd.DataFrame({

        "Feature": sample.columns,

        "Value": sample.iloc[0].values,

        "Contribution": shap_values

    })

    explanation_df["Absolute"] = explanation_df[
        "Contribution"
    ].abs()

    explanation_df = explanation_df.sort_values(
        by="Absolute",
        ascending=False
    )

    return explanation_df.head(top_n)

**Rule Dictionary**

In [58]:
FEATURE_RULES = {

    "max_price":
        "The current maximum market price strongly influenced the prediction.",

    "min_price":
        "The current minimum market price significantly affected the forecast.",

    "lag_1":
        "The previous week's market price contributed to the prediction.",

    "lag_2":
        "Recent historical prices influenced the expected market trend.",

    "lag_4":
        "The one-month historical price pattern contributed to the prediction.",

    "lag_12":
        "Long-term historical price behaviour had a minor influence.",

    "price_4w_avg":
        "The four-week moving average indicates recent market behaviour.",

    "price_8w_avg":
        "The eight-week moving average reflects medium-term market trends.",

    "production_total":
        "Production volume had a limited impact on the predicted price.",

    "price_change":
        "Recent price fluctuations slightly influenced the prediction.",

    "rolling_std_4":
        "Market price volatility contributed to the prediction.",

    "season_Yala":
        "Seasonal cultivation patterns slightly influenced the forecast."
}

**Human Explanation Generator**

In [73]:
def generate_explanation(sample,
                         prediction,
                         previous_price,
                         shap_values):

    trend = classify_trend(
        previous_price,
        prediction
    )

    confidence = estimate_confidence(
        shap_values
    )

    top_features = get_top_features(
        sample,
        shap_values,
        top_n=3
    )

    dynamic_reasons = generate_dynamic_reasons(
    sample,
    shap_values,
    top_n=5
    )



    return {

        "prediction": round(float(prediction),2),

        "trend": trend,

        "confidence": confidence,

        "top_features": top_features,

        "reasons": dynamic_reasons

    }

**Generate Explanation**

In [75]:
sample_index = 100

sample = X.iloc[[sample_index]]

prediction = model.predict(sample)[0]

previous_price = sample["lag_1"].values[0]

local_shap = explainer.shap_values(sample)[0]

result = generate_explanation(

    sample,

    prediction,

    previous_price,

    local_shap

)

print("Explanation Generated Successfully")

Explanation Generated Successfully


**Display Explanation**

In [76]:
print("="*60)

print("PADDY PRICE EXPLANATION")

print("="*60)

print(f"Predicted Price : {result['prediction']:.2f} LKR/kg")

print()

print(f"Trend : {result['trend']}")

print()

print(f"Confidence : {result['confidence']}")

print()

print("Main Reasons")

for reason in result["reasons"]:

    print(f"• {reason}")

PADDY PRICE EXPLANATION
Predicted Price : 37.05 LKR/kg

Trend : Stable

Confidence : High

Main Reasons
• The maximum market price is 38.00 LKR/kg. This reduced the predicted average price by approximately 13.44 LKR.
• The minimum market price is 36.00 LKR/kg. This reduced the predicted average price by approximately 5.69 LKR.
• The previous week's average price was 36.40 LKR/kg. It reduced today's prediction by 0.43 LKR.
• The eight-week moving average is 46.49 LKR/kg. It reduced the prediction by 0.34 LKR.
• The four-week moving average is 43.00 LKR/kg. It reduced the prediction by 0.19 LKR.


**Export Explanation**

In [77]:
output = {

    "prediction": result["prediction"],

    "trend": result["trend"],

    "confidence": result["confidence"],

    "reasons": result["reasons"]

}

with open(

    os.path.join(

        OUTPUT_DIR,

        "human_explanation.json"

    ),

    "w"

) as f:

    json.dump(

        output,

        f,

        indent=4

    )

print("Human explanation saved successfully.")

Human explanation saved successfully.


**Dynamic Feature Explanation Function**

In [78]:
def explain_feature(feature, value, shap_value):

    direction = "increased" if shap_value > 0 else "reduced"

    impact = abs(shap_value)

    if feature == "max_price":

        return (
            f"The maximum market price is {value:.2f} LKR/kg. "
            f"This {direction} the predicted average price by "
            f"approximately {impact:.2f} LKR."
        )

    elif feature == "min_price":

        return (
            f"The minimum market price is {value:.2f} LKR/kg. "
            f"This {direction} the predicted average price by "
            f"approximately {impact:.2f} LKR."
        )

    elif feature == "lag_1":

        return (
            f"The previous week's average price was {value:.2f} LKR/kg. "
            f"It {direction} today's prediction by "
            f"{impact:.2f} LKR."
        )

    elif feature == "price_4w_avg":

        return (
            f"The four-week moving average is {value:.2f} LKR/kg. "
            f"It {direction} the prediction by "
            f"{impact:.2f} LKR."
        )

    elif feature == "price_8w_avg":

        return (
            f"The eight-week moving average is {value:.2f} LKR/kg. "
            f"It {direction} the prediction by "
            f"{impact:.2f} LKR."
        )

    elif feature == "production_total":

        return (
            f"The production volume is {int(value):,}. "
            f"It {direction} the prediction by "
            f"{impact:.2f} LKR."
        )

    elif feature == "price_change":

        return (
            f"The recent price change is {value:.2f} LKR/kg. "
            f"It {direction} the prediction by "
            f"{impact:.2f} LKR."
        )

    else:

        return (
            f"{feature} = {value} "
            f"{direction} the prediction by "
            f"{impact:.2f} LKR."
        )

**Generate Dynamic Reasons**

In [79]:
def generate_dynamic_reasons(sample,
                             shap_values,
                             top_n=5):

    explanation_df = pd.DataFrame({

        "Feature": sample.columns,

        "Value": sample.iloc[0].values,

        "Contribution": shap_values

    })

    explanation_df["Absolute"] = explanation_df[
        "Contribution"
    ].abs()

    explanation_df = explanation_df.sort_values(
        by="Absolute",
        ascending=False
    )

    reasons = []

    for _, row in explanation_df.head(top_n).iterrows():

        reasons.append(

            explain_feature(

                row["Feature"],

                row["Value"],

                row["Contribution"]

            )

        )

    return reasons

**Test Dynamic Explanation**

In [80]:
print("="*70)
print("PADDY PRICE EXPLANATION")
print("="*70)

print(f"Predicted Price : {result['prediction']:.2f} LKR/kg")

print(f"Trend           : {result['trend']}")

print(f"Confidence      : {result['confidence']}")

print("\nExplanation")

for i, reason in enumerate(result["reasons"], start=1):

    print(f"{i}. {reason}")

PADDY PRICE EXPLANATION
Predicted Price : 37.05 LKR/kg
Trend           : Stable
Confidence      : High

Explanation
1. The maximum market price is 38.00 LKR/kg. This reduced the predicted average price by approximately 13.44 LKR.
2. The minimum market price is 36.00 LKR/kg. This reduced the predicted average price by approximately 5.69 LKR.
3. The previous week's average price was 36.40 LKR/kg. It reduced today's prediction by 0.43 LKR.
4. The eight-week moving average is 46.49 LKR/kg. It reduced the prediction by 0.34 LKR.
5. The four-week moving average is 43.00 LKR/kg. It reduced the prediction by 0.19 LKR.


**Market Outlook Generator**

In [81]:
def generate_market_outlook(trend):

    outlook = {

        "Strong Increase":
            "Market conditions indicate a noticeable upward price movement.",

        "Increase":
            "The market shows signs of gradual price improvement.",

        "Stable":
            "Market prices appear relatively stable with only minor fluctuations.",

        "Decrease":
            "The market indicates a gradual downward price trend.",

        "Strong Decrease":
            "The market is experiencing a significant decline in prices."

    }

    return outlook.get(
        trend,
        "Market outlook unavailable."
    )

**Recommendation Generator**

In [82]:
def generate_recommendation(
    trend,
    confidence
):

    recommendations = {

        "Strong Increase":
            "Prices are forecast to increase considerably. Farmers with adequate storage facilities may consider monitoring the market before deciding when to sell.",

        "Increase":
            "A moderate price increase is expected. Monitoring market prices over the coming weeks may help support informed selling decisions.",

        "Stable":
            "Prices are expected to remain relatively stable. Farmers may continue with their planned marketing strategy while observing future market updates.",

        "Decrease":
            "Prices are forecast to decline gradually. Farmers may wish to review current market conditions when planning sales.",

        "Strong Decrease":
            "A significant decline in prices is forecast. Farmers should carefully monitor market developments and consider available marketing options."

    }

    recommendation = recommendations.get(
        trend,
        "No recommendation available."
    )

    if confidence == "Low":

        recommendation += (
            " Prediction confidence is low, therefore additional market information should also be considered."
        )

    return recommendation

**Risk Assessment**

In [83]:
def assess_risk(
    trend,
    confidence
):

    if trend in ["Strong Increase", "Strong Decrease"]:

        if confidence == "High":

            return "High"

        return "Medium"

    elif trend in ["Increase", "Decrease"]:

        return "Medium"

    return "Low"

**Prediction Summary**

In [84]:
def generate_summary(result):

    return (

        f"The model predicts an average paddy price of "
        f"{result['prediction']:.2f} LKR/kg. "

        f"The expected market trend is "

        f"'{result['trend']}'. "

        f"The explanation is supported with "

        f"{result['confidence'].lower()} confidence."

    )

**Decision Support Report**

In [85]:
market_outlook = generate_market_outlook(
    result["trend"]
)

recommendation = generate_recommendation(
    result["trend"],
    result["confidence"]
)

risk = assess_risk(
    result["trend"],
    result["confidence"]
)

summary = generate_summary(result)

decision_support = {

    "prediction": result["prediction"],

    "trend": result["trend"],

    "confidence": result["confidence"],

    "risk_level": risk,

    "market_outlook": market_outlook,

    "recommendation": recommendation,

    "summary": summary,

    "reasons": result["reasons"]

}

**Display Report**

In [86]:
print("="*70)
print("AI DECISION SUPPORT REPORT")
print("="*70)

print(f"Predicted Price : {decision_support['prediction']:.2f} LKR/kg")

print(f"Trend           : {decision_support['trend']}")

print(f"Confidence      : {decision_support['confidence']}")

print(f"Risk Level      : {decision_support['risk_level']}")

print()

print("Prediction Summary")

print(decision_support["summary"])

print()

print("Market Outlook")

print(decision_support["market_outlook"])

print()

print("Recommendation")

print(decision_support["recommendation"])

print()

print("Key Reasons")

for reason in decision_support["reasons"]:

    print(f"• {reason}")

AI DECISION SUPPORT REPORT
Predicted Price : 37.05 LKR/kg
Trend           : Stable
Confidence      : High
Risk Level      : Low

Prediction Summary
The model predicts an average paddy price of 37.05 LKR/kg. The expected market trend is 'Stable'. The explanation is supported with high confidence.

Market Outlook
Market prices appear relatively stable with only minor fluctuations.

Recommendation
Prices are expected to remain relatively stable. Farmers may continue with their planned marketing strategy while observing future market updates.

Key Reasons
• The maximum market price is 38.00 LKR/kg. This reduced the predicted average price by approximately 13.44 LKR.
• The minimum market price is 36.00 LKR/kg. This reduced the predicted average price by approximately 5.69 LKR.
• The previous week's average price was 36.40 LKR/kg. It reduced today's prediction by 0.43 LKR.
• The eight-week moving average is 46.49 LKR/kg. It reduced the prediction by 0.34 LKR.
• The four-week moving average i